In [ ]:
#图3c 生态型富集
import pandas as pd
import os
import scipy.stats as st
import numpy as np
import statsmodels.stats.multitest as smt
import seaborn as sns
import matplotlib.pyplot as plt
from statannotations.Annotator import Annotator
import re
import scipy.stats as stats
from statsmodels.stats.multitest import fdrcorrection
import numpy as np
import pandas as pd

def calculate_levins_bj(df):
    """
    计算 Levins' Niche Breadth (Bj)
    公式 Eq(1): Bj = 1 / sum(Pij^2)
    输入 df 必须是相对丰度矩阵 (Relative Abundance Matrix)
    """
    # 确保是浮点数
    matrix = df.astype(float)
    # 计算平方和
    sum_sq = (matrix ** 2).sum(axis=1)
    # 避免除以0
    bj = 1.0 / sum_sq.replace(0, np.nan)
    return bj.fillna(0)

def classify_generalist_specialist(relative_abundance_matrix, num_permutations=1000, seed=42):
    """
    置换检验分类广适/专适类群
    """
    # 1. 准备数据
    df = relative_abundance_matrix.fillna(0).astype(float)
    
    # 2. 计算观察值的 Bj (Observed)
    obs_bj = calculate_levins_bj(df)
    
    # 3. 置换检验 (Permutation Test)
    # "shuffling the OTU relative abundance matrix" -> 全矩阵打乱
    rng = np.random.default_rng(seed=seed)
    
    # 记录置换后的Bj大于或小于观察值的次数
    count_obs_greater = pd.Series(0, index=df.index)
    count_obs_less = pd.Series(0, index=df.index)
    
    # 将矩阵展平以便全矩阵打乱
    matrix_values_flat = df.values.flatten()
    original_shape = df.shape
    
    for i in range(num_permutations):
        # 复制并打乱整个数值池
        shuffled_flat = matrix_values_flat.copy()
        rng.shuffle(shuffled_flat)
        
        # 重塑回矩阵
        shuffled_df = pd.DataFrame(
            shuffled_flat.reshape(original_shape),
            index=df.index,
            columns=df.columns
        )
        
        # 计算置换数据的 Bj (Expected/Null distribution)
        perm_bj = calculate_levins_bj(shuffled_df)
        
        # 统计
        count_obs_greater += (obs_bj > perm_bj).astype(int)
        count_obs_less += (obs_bj < perm_bj).astype(int)
    
    # 4. 计算结果
    results = pd.DataFrame(index=df.index)
    results['Bj_Observed'] = obs_bj
    
    # 计算 P 值 (以 1 - 概率 的形式，使其符合 < 0.05 的习惯)
    # 如果 observed > expected 的概率是 95%，则 p_gen = 0.05
    results['p_generalist'] = 1 - (count_obs_greater / num_permutations)
    results['p_specialist'] = 1 - (count_obs_less / num_permutations)
    
    # 5. 分类 (Cutoff: P < 0.05)
    results['Ecotype'] = 'Neutral' # 默认为中性
    
    # 标记 Generalist
    results.loc[results['p_generalist'] < 0.05, 'Ecotype'] = 'Generalist'
    
    # 标记 Specialist
    results.loc[results['p_specialist'] < 0.05, 'Ecotype'] = 'Specialist'
    
    return results


#  细菌生态型
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
bacteria_ASV = pd.read_csv("/mnt/d/study/master/meiji/bacteria_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 bacteria_ASV DataFrame 中选择指定的列
b_ASV = bacteria_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
b_ASV.set_index(keys="ASV", inplace=True)
b_tax = bacteria_ASV.iloc[:, 1:9].copy()
b_tax.set_index(keys="ASV", inplace=True)
print(len(b_ASV))
# 计算 ASV 相对丰度
b_ASV_df = b_ASV.copy()

#按种
b_species = b_ASV.join(b_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
b_species = b_species.groupby('Species')[sample_ids].sum()
#排序
b_species = b_species.sort_index()
b_species_df = b_species.copy()
print(len(b_species_df))
# 将当前的 ASV 索引重置为一个普通列。
b_tax_reset_index = b_tax.reset_index()
# 将 'Species' 列设置为新的索引。
b_tax_species = b_tax_reset_index.groupby('Species').first()
b_tax_species = b_tax_species.drop(columns=['ASV'])
b_tax_species = b_tax_species.sort_index()
# 进行替换操作
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__unclassified_k__norank_d__Bacteria', 'Unclassified')
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__SAR324_cladeMarine_group_B', 'SAR324')
# 去除 'p__' 前缀
b_tax_species['Phylum'] = b_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
b_unique_species_names = b_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
b_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(b_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 b_species 的索引
b_species = b_species.rename(index= b_species_to_numeric_id)
# 转换 b_tax_species 的索引
b_tax_species = b_tax_species.rename(index= b_species_to_numeric_id)
b_species_df = b_species.copy()
print(len(b_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
b_species_rel = b_species_df.div(b_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
b_rel_long = b_species_rel.stack().reset_index()
b_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
b_rel_long['Origin'] = b_rel_long['Sample'].map(sample_origins_map)
b_rel_long['Niche'] = b_rel_long['Sample'].map(sample_niches_map)
b_species_total_counts = b_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
b_total_counts_across_all_species = b_species_total_counts.sum() # 计算所有 species 的总丰度
b_species_relative_abundance = b_species_total_counts / b_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
b_species_above_overall_threshold = b_species_relative_abundance[b_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
b_rel_long_filtered_by_overall_abundance = b_rel_long[b_rel_long['species'].isin(b_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
b_species_origin_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
b_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
b_species_in_origin = b_species_origin_mean_filtered[b_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
b_species_origin_count = b_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
b_species_niche_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
b_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
b_species_in_niche = b_species_niche_mean_filtered[b_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
b_species_niche_count = b_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
b_species_counts_combined = pd.merge(b_species_origin_count, b_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 4
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
b_core_species_names = b_species_counts_combined[
    (b_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (b_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
b_core_abundance = b_species_rel.loc[b_core_species_names]
b_core_species_raw_counts = b_species.loc[b_core_species_names]
print(len(b_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
b_median_relative_abundance = b_species_relative_abundance.median()
print(b_median_relative_abundance)
# 定义稀有类群的丰度阈值 
b_rare_threshold = 0.00001
# 筛选出稀有 species 
b_rare_species = b_species_relative_abundance[b_species_relative_abundance < b_rare_threshold]
# 获取稀有 species 的名称（索引）
b_rare_species_names = b_rare_species.index.tolist()
len(b_rare_species_names)
b_species_db_transposed = b_species_df.T 
b_speciesdfmelt = b_species_db_transposed.stack().reset_index()# 长宽表格转换
b_speciesdfmelt.columns = ['Sample', 'species', 'value']
b_speciesdfmelt = b_speciesdfmelt[b_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
b_control_species = b_speciesdfmelt.copy()
b_control_species['Group'] = b_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
b_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
b_controlspecies_unique = b_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
b_controlspecies_unique = b_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
b_controlspecies_unique = b_controlspecies_unique[b_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
b_unique_species_names_single_group = b_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
b_set_rare_species = set(b_rare_species_names)
b_set_unique_species_single_group = set(b_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
b_rare_unique_species_names = list(b_set_rare_species.intersection(b_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
b_rare_unique_abundance = b_species_df.loc[b_rare_unique_species_names]
b_rare_unique_species_raw_counts = b_species.loc[b_rare_unique_species_names]
print(len(b_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度计算
num_permutations = 1000 
b_niche_breadth_results = classify_generalist_specialist(
    b_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (前5个 species) ")
print(b_niche_breadth_results.head())
b_generalists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
b_specialists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(b_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(b_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_generalists_species:
    print("无广适类群 species 数据。")
else:
    b_generalists_raw_counts = b_species.loc[b_generalists_species]
    b_generalists_abundance = b_species_rel.loc[b_generalists_species]
    print(b_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_specialists_species:
    print("无专适类群 species 数据。")
else:
    b_specialists_raw_counts = b_species.loc[b_specialists_species]
    b_specialists_abundance = b_species_rel.loc[b_specialists_species]
print(b_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

output_dir = r'/mnt/d/study/'
file_path_b_gen = os.path.join(output_dir, 'b_generalists_raw_counts.csv') 
b_generalists_raw_counts.to_csv(file_path_b_gen, index=True)
file_path_b_spe = os.path.join(output_dir, 'b_specialists_raw_counts.csv') 
b_specialists_raw_counts.to_csv(file_path_b_spe, index=True)

# 加载相对丰度、丰度物种、稀有物种、广适种和狭适种数据
b_df_OTU_rel = b_species_rel 
b_df_abun = b_core_abundance 
b_df_rare = b_rare_unique_abundance
b_df_gen = b_generalists_abundance
b_df_spe = b_specialists_abundance
# 加载生态系统数据
b_ecosystem = metadata
b_ecosystem = b_ecosystem.iloc[:, :2]
b_ecosystem.set_index(b_ecosystem.columns[0], inplace=True)
b_taxon = b_tax_species

#  门水平的分组富集分析 
# 计算每个门（Phylum）的预期比例 (expected proportion)
b_prop_exp = b_taxon['Phylum'].value_counts(normalize=True).to_frame(name='Proportion')
# 计算每个分组中每个门（Phylum）的观察计数 (observed count)
b_taxon_abun = b_taxon[b_taxon.index.isin(b_df_abun.index)]
b_taxon_rare = b_taxon[b_taxon.index.isin(b_df_rare.index)]
b_taxon_gen = b_taxon[b_taxon.index.isin(b_df_gen.index)]
b_taxon_spe = b_taxon[b_taxon.index.isin(b_df_spe.index)]
b_count_abun = b_taxon_abun['Phylum'].value_counts(normalize=False).to_frame(name='Count')
b_count_rare = b_taxon_rare['Phylum'].value_counts(normalize=False).to_frame(name='Count')
b_count_gen = b_taxon_gen['Phylum'].value_counts(normalize=False).to_frame(name='Count')
b_count_spe = b_taxon_spe['Phylum'].value_counts(normalize=False).to_frame(name='Count')
#定义一个函数，基于二项分布执行富集分析
def enrichment_analysis(obs_ct_df, exp_prop_df):
    """
    基于二项分布计算门的富集指数。
    富集指数的计算公式为 (观察计数 - 预期计数) / 预期计数的标准差。
    （这里是 Z-score 形式的富集指数，其中标准差为 sqrt(N * p * (1-p))）
    参数:
        obs_ct_df (DataFrame): 包含每个门观察计数的 DataFrame，索引是门名称，列名为 'Count'。
        exp_prop_df (DataFrame): 包含每个门预期比例的 DataFrame，索引是门名称，列名为 'Proportion'。
    返回:
        DataFrame: 包含每个门的富集指数，索引是门名称，列名为 'Enrichment index'。
    """
    i_dict = {} # 用于存储每个门的富集指数
    N = obs_ct_df['Count'].sum() # 当前分组中OTU的总数 (即样本总数)
    # 遍历观察计数 DataFrame 中的每个门
    for index, row in obs_ct_df.iterrows():
        n = row['Count'] # 当前门在当前分组中的观察计数
        # 获取当前门在所有OTU中的预期比例
        # 使用 .loc[] 或 .at[] 更安全地访问单个值，避免 ._get_value 这种内部方法
        # 确保该门在 exp_prop_df 中存在，否则会报错
        if index in exp_prop_df.index:
            p = exp_prop_df.loc[index, 'Proportion']
        else:
            p = 0 # 如果该门在总样本中不存在，则预期比例为0
            # 或者可以跳过该门或发出警告，取决于分析需求
            # continue
        # 避免除以零或 sqrt(负数)，尤其当 p 接近 0 或 1 时
        if p == 0 or p == 1:
            i = 0 # 或 np.inf / -np.inf，取决于定义，这里设为0表示无波动
        else:
            # 计算富集指数 (Z-score 形式)
            # 预期计数 = N * p
            # 标准差 = sqrt(N * p * (1 - p))
            denominator = np.sqrt(N * p * (1 - p))
            if denominator == 0: # 再次检查分母是否为零
                i = 0
            else:
                i = (n - N * p) / denominator
        i_dict[index] = i
    # 将结果字典转换为 DataFrame
    i_df = pd.DataFrame.from_dict(i_dict, orient='index',columns=['Enrichment index'])
    return i_df
# 对每个分组执行富集分析
b_abun_enrich = enrichment_analysis(b_count_abun, b_prop_exp)
b_rare_enrich = enrichment_analysis(b_count_rare, b_prop_exp)
b_gen_enrich = enrichment_analysis(b_count_gen, b_prop_exp)
b_spe_enrich = enrichment_analysis(b_count_spe, b_prop_exp)
# 准备一个 DataFrame，包含门、富集指数、OTU计数和分组，以便合并所有分组的数据
# 为“丰度物种”合并富集指数和计数，重置索引，排序，并添加 'Ecotype' 列
b_abun_enrich_count = pd.merge(b_abun_enrich, b_count_abun, left_index = True, right_index = True)
b_abun_enrich_count = b_abun_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
b_abun_enrich_count['Ecotype'] = 'Cores'
# 为“稀有物种”重复上述步骤
b_rare_enrich_count = pd.merge(b_rare_enrich, b_count_rare, left_index = True, right_index = True)
b_rare_enrich_count = b_rare_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
b_rare_enrich_count['Ecotype'] = 'Uniques'
# 为“广适种”重复上述步骤
b_gen_enrich_count = pd.merge(b_gen_enrich, b_count_gen, left_index = True, right_index = True)
b_gen_enrich_count = b_gen_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
b_gen_enrich_count['Ecotype'] = 'Generalists'
# 为“狭适种”重复上述步骤
b_spe_enrich_count = pd.merge(b_spe_enrich, b_count_spe, left_index = True, right_index = True)
b_spe_enrich_count = b_spe_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
b_spe_enrich_count['Ecotype'] = 'Specialists'
# 将所有分组富集结果合并到一个大的 DataFrame 中
b_all_enrich_count = pd.concat([b_abun_enrich_count, b_rare_enrich_count, b_gen_enrich_count, b_spe_enrich_count])
# 打印合并后的 DataFrame，以查看结果（可选）
# print(all_enrich_count)
# 定义模糊门模式 (与之前相同)
ambiguous_patterns = re.compile(
   r"uncultured|unclassified|norank|chloroplast|unknown|SAR324|MBNT15|RCP2", 
   flags=re.IGNORECASE
)
# 去除 'p__' 前缀
b_all_enrich_count['Phylum'] = b_all_enrich_count['Phylum'].astype(str).str.replace("p__", "")
# 过滤掉匹配模糊模式的门
# 使用 apply 和 lambda 函数来检查每个门名是否包含模糊模式
b_all_enrich_count = b_all_enrich_count[~b_all_enrich_count['Phylum'].apply(lambda x: bool(ambiguous_patterns.search(str(x))))]
#  绘制气泡图 
# 定义分组与颜色之间的映射关系
b_group_colors = {
    'Cores': '#2471A3', # 蓝色
    'Uniques': '#D4AC0D',      # 金色
    'Generalists': '#EC7063',    # 红色
    'Specialists': '#82E0AA'     # 浅绿色
}
# 设置 Matplotlib 图的尺寸（宽度，高度）
plt.rcParams["figure.figsize"] = (5, 5)
# 使用 seaborn 绘制散点图（作为气泡图）
bubble_plot = sns.scatterplot(
    data= b_all_enrich_count,         # 输入数据 DataFrame
    x='Enrichment index',          # X轴：富集指数
    y='Phylum',                    # Y轴：门名称
    size='Count',                  # 气泡大小由 'Count'（OTU数量）决定
    hue='Ecotype',                 # 气泡颜色由 'Ecotype'（分组）决定
    palette=b_group_colors,          # 使用预定义的颜色映射
    sizes=(20, 250),               # 气泡大小的范围 (最小值, 最大值)
    alpha=1,                       # 气泡透明度
    edgecolor='black',             # 气泡边缘颜色
    linewidth=0.5                  # 气泡边缘线宽
)
# 添加垂直参考线，表示富集指数的阈值
# 通常，富集指数大于 2 或小于 -2 被认为是显著的
plt.axvline(x=-2, color='grey', linestyle='--') # 负富集（稀释）阈值
plt.axvline(x=2, color='grey', linestyle='--')  # 正富集阈值
# 设置图表标题、X轴和Y轴标签
plt.title('Bacterial phyla representation within ecotype', size = 13)
plt.xlabel('Enrichment index', size = 13)
plt.ylabel('Phylum', size = 13)
# 调整图例位置，使其不与图表内容重叠
# bbox_to_anchor=(1.45, 0.63): 将图例锚点移动到轴外部
plt.legend(bbox_to_anchor=(1.45, 0.63))
# 设置X轴的显示范围
plt.xlim(-8.2,8.2)
# 定义保存图表的输出目录
output_dir = '/mnt/d/study//'
# 如果目录不存在，则创建它
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建输出目录: {output_dir}")
# 保存图表
# bbox_inches='tight' 确保所有图表元素都包含在保存的图像中
# dpi=600 设置图像的分辨率
plt.savefig(os.path.join(output_dir, 'ecotype_phyla_enrich_bacteria.png'), bbox_inches='tight', dpi=600)
# 显示图表
plt.show()



#  真菌生态型
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
fungi_ASV = pd.read_csv("/mnt/d/study/master/meiji/fungi_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 fungi_ASV DataFrame 中选择指定的列
f_ASV = fungi_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
f_ASV.set_index(keys="ASV", inplace=True)
f_tax = fungi_ASV.iloc[:, 1:9].copy()
f_tax.set_index(keys="ASV", inplace=True)
print(len(f_ASV))
# 计算 ASV 相对丰度
f_ASV_df = f_ASV.copy()

#按种
f_species = f_ASV.join(f_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
f_species = f_species.groupby('Species')[sample_ids].sum()
#排序
f_species = f_species.sort_index()
# 将当前的 ASV 索引重置为一个普通列。
f_tax_reset_index = f_tax.reset_index()
# 将 'Species' 列设置为新的索引。
f_tax_species = f_tax_reset_index.groupby('Species').first()
f_tax_species = f_tax_species.drop(columns=['ASV'])
f_tax_species = f_tax_species.sort_index()
# 进行替换操作
f_tax_species['Phylum'] = f_tax_species['Phylum'].replace('p__unclassified_k__Fungi', 'Unclassified')
# 去除 'p__' 前缀
f_tax_species['Phylum'] = f_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
f_unique_species_names = f_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
f_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(f_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 f_species 的索引
f_species = f_species.rename(index= f_species_to_numeric_id)
# 转换 f_tax_species 的索引
f_tax_species = f_tax_species.rename(index= f_species_to_numeric_id)
f_species_df = f_species.copy()
print(len(f_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
f_species_rel = f_species_df.div(f_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
f_rel_long = f_species_rel.stack().reset_index()
f_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
f_rel_long['Origin'] = f_rel_long['Sample'].map(sample_origins_map)
f_rel_long['Niche'] = f_rel_long['Sample'].map(sample_niches_map)
f_species_total_counts = f_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
f_total_counts_across_all_species = f_species_total_counts.sum() # 计算所有 species 的总丰度
f_species_relative_abundance = f_species_total_counts / f_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
f_species_above_overall_threshold = f_species_relative_abundance[f_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
f_rel_long_filtered_by_overall_abundance = f_rel_long[f_rel_long['species'].isin(f_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
f_species_origin_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
f_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
f_species_in_origin = f_species_origin_mean_filtered[f_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
f_species_origin_count = f_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
f_species_niche_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
f_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
f_species_in_niche = f_species_niche_mean_filtered[f_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
f_species_niche_count = f_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
f_species_counts_combined = pd.merge(f_species_origin_count, f_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 4
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
f_core_species_names = f_species_counts_combined[
    (f_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (f_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
f_core_abundance = f_species_rel.loc[f_core_species_names]
f_core_species_raw_counts = f_species.loc[f_core_species_names]
print(len(f_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
f_median_relative_abundance = f_species_relative_abundance.median()
print(f_median_relative_abundance)
# 定义稀有类群的丰度阈值 
f_rare_threshold = 0.000008
# 筛选出稀有 species 
f_rare_species = f_species_relative_abundance[f_species_relative_abundance < f_rare_threshold]
# 获取稀有 species 的名称（索引）
f_rare_species_names = f_rare_species.index.tolist()
len(f_rare_species_names)
f_species_df_transposed = f_species_df.T 
f_speciesdfmelt = f_species_df_transposed.stack().reset_index()# 长宽表格转换
f_speciesdfmelt.columns = ['Sample', 'species', 'value']
f_speciesdfmelt = f_speciesdfmelt[f_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
f_control_species = f_speciesdfmelt.copy()
f_control_species['Group'] = f_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
f_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
f_controlspecies_unique = f_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
f_controlspecies_unique = f_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
f_controlspecies_unique = f_controlspecies_unique[f_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
f_unique_species_names_single_group = f_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
f_set_rare_species = set(f_rare_species_names)
f_set_unique_species_single_group = set(f_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
f_rare_unique_species_names = list(f_set_rare_species.intersection(f_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
f_rare_unique_abundance = f_species_df.loc[f_rare_unique_species_names]
f_rare_unique_species_raw_counts = f_species.loc[f_rare_unique_species_names]
print(len(f_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度计算
num_permutations = 1000 
f_niche_breadth_results = classify_generalist_specialist(
    f_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (前5个 species) ")
print(f_niche_breadth_results.head())
f_generalists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
f_specialists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(f_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(f_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_generalists_species:
    print("无广适类群 species 数据。")
else:
    f_generalists_raw_counts = f_species.loc[f_generalists_species]
    f_generalists_abundance = f_species_rel.loc[f_generalists_species]
    print(f_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_specialists_species:
    print("无专适类群 species 数据。")
else:
    f_specialists_raw_counts = f_species.loc[f_specialists_species]
    f_specialists_abundance = f_species_rel.loc[f_specialists_species]
print(f_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

file_path_f_gen = os.path.join(output_dir, 'f_generalists_raw_counts.csv') 
f_generalists_raw_counts.to_csv(file_path_f_gen, index=True)
file_path_f_spe = os.path.join(output_dir, 'f_specialists_raw_counts.csv') 
f_specialists_raw_counts.to_csv(file_path_f_spe, index=True)

# 加载相对丰度、丰度物种、稀有物种、广适种和狭适种数据
f_df_OTU_rel = f_species_rel 
f_df_abun = f_core_abundance 
f_df_rare = f_rare_unique_abundance
f_df_gen = f_generalists_abundance
f_df_spe = f_specialists_abundance
# 加载生态系统数据
f_ecosystem = metadata
f_ecosystem = f_ecosystem.iloc[:, :2]
f_ecosystem.set_index(f_ecosystem.columns[0], inplace=True)
f_taxon = f_tax_species

#  门水平的分组富集分析 
# 计算每个门（Phylum）的预期比例 (expected proportion)
f_prop_exp = f_taxon['Phylum'].value_counts(normalize=True).to_frame(name='Proportion')
# 计算每个分组中每个门（Phylum）的观察计数 (observed count)
f_taxon_abun = f_taxon[f_taxon.index.isin(f_df_abun.index)]
f_taxon_rare = f_taxon[f_taxon.index.isin(f_df_rare.index)]
f_taxon_gen = f_taxon[f_taxon.index.isin(f_df_gen.index)]
f_taxon_spe = f_taxon[f_taxon.index.isin(f_df_spe.index)]
f_count_abun = f_taxon_abun['Phylum'].value_counts(normalize=False).to_frame(name='Count')
f_count_rare = f_taxon_rare['Phylum'].value_counts(normalize=False).to_frame(name='Count')
f_count_gen = f_taxon_gen['Phylum'].value_counts(normalize=False).to_frame(name='Count')
f_count_spe = f_taxon_spe['Phylum'].value_counts(normalize=False).to_frame(name='Count')
#定义一个函数，基于二项分布执行富集分析
def enrichment_analysis(obs_ct_df, exp_prop_df):
    """
    基于二项分布计算门的富集指数。
    富集指数的计算公式为 (观察计数 - 预期计数) / 预期计数的标准差。
    （这里是 Z-score 形式的富集指数，其中标准差为 sqrt(N * p * (1-p))）
    参数:
        obs_ct_df (DataFrame): 包含每个门观察计数的 DataFrame，索引是门名称，列名为 'Count'。
        exp_prop_df (DataFrame): 包含每个门预期比例的 DataFrame，索引是门名称，列名为 'Proportion'。
    返回:
        DataFrame: 包含每个门的富集指数，索引是门名称，列名为 'Enrichment index'。
    """
    i_dict = {} # 用于存储每个门的富集指数
    N = obs_ct_df['Count'].sum() # 当前分组中OTU的总数 (即样本总数)
    # 遍历观察计数 DataFrame 中的每个门
    for index, row in obs_ct_df.iterrows():
        n = row['Count'] # 当前门在当前分组中的观察计数
        # 获取当前门在所有OTU中的预期比例
        # 使用 .loc[] 或 .at[] 更安全地访问单个值，避免 ._get_value 这种内部方法
        # 确保该门在 exp_prop_df 中存在，否则会报错
        if index in exp_prop_df.index:
            p = exp_prop_df.loc[index, 'Proportion']
        else:
            p = 0 # 如果该门在总样本中不存在，则预期比例为0
            # 或者可以跳过该门或发出警告，取决于分析需求
            # continue
        # 避免除以零或 sqrt(负数)，尤其当 p 接近 0 或 1 时
        if p == 0 or p == 1:
            i = 0 # 或 np.inf / -np.inf，取决于定义，这里设为0表示无波动
        else:
            # 计算富集指数 (Z-score 形式)
            # 预期计数 = N * p
            # 标准差 = sqrt(N * p * (1 - p))
            denominator = np.sqrt(N * p * (1 - p))
            if denominator == 0: # 再次检查分母是否为零
                i = 0
            else:
                i = (n - N * p) / denominator
        i_dict[index] = i
    # 将结果字典转换为 DataFrame
    i_df = pd.DataFrame.from_dict(i_dict, orient='index',columns=['Enrichment index'])
    return i_df
# 对每个分组执行富集分析
f_abun_enrich = enrichment_analysis(f_count_abun, f_prop_exp)
f_rare_enrich = enrichment_analysis(f_count_rare, f_prop_exp)
f_gen_enrich = enrichment_analysis(f_count_gen, f_prop_exp)
f_spe_enrich = enrichment_analysis(f_count_spe, f_prop_exp)
# 准备一个 DataFrame，包含门、富集指数、OTU计数和分组，以便合并所有分组的数据
# 为“丰度物种”合并富集指数和计数，重置索引，排序，并添加 'Ecotype' 列
f_abun_enrich_count = pd.merge(f_abun_enrich, f_count_abun, left_index = True, right_index = True)
f_abun_enrich_count = f_abun_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
f_abun_enrich_count['Ecotype'] = 'Cores'
# 为“稀有物种”重复上述步骤
f_rare_enrich_count = pd.merge(f_rare_enrich, f_count_rare, left_index = True, right_index = True)
f_rare_enrich_count = f_rare_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
f_rare_enrich_count['Ecotype'] = 'Uniques'
# 为“广适种”重复上述步骤
f_gen_enrich_count = pd.merge(f_gen_enrich, f_count_gen, left_index = True, right_index = True)
f_gen_enrich_count = f_gen_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
f_gen_enrich_count['Ecotype'] = 'Generalists'
# 为“狭适种”重复上述步骤
f_spe_enrich_count = pd.merge(f_spe_enrich, f_count_spe, left_index = True, right_index = True)
f_spe_enrich_count = f_spe_enrich_count.reset_index(names=['Phylum']).sort_values(by="Enrichment index", ascending=False)
f_spe_enrich_count['Ecotype'] = 'Specialists'
# 将所有分组富集结果合并到一个大的 DataFrame 中
f_all_enrich_count = pd.concat([f_abun_enrich_count, f_rare_enrich_count, f_gen_enrich_count, f_spe_enrich_count])
# 打印合并后的 DataFrame，以查看结果（可选）
# print(all_enrich_count)
# 定义模糊门模式 (与之前相同)
ambiguous_patterns = re.compile(
   r"uncultured|unclassified|norank|chloroplast|unknown|SAR324|MBNT15|RCP2", 
   flags=re.IGNORECASE
)
# 去除 'p__' 前缀
f_all_enrich_count['Phylum'] = f_all_enrich_count['Phylum'].astype(str).str.replace("p__", "")
# 过滤掉匹配模糊模式的门
# 使用 apply 和 lambda 函数来检查每个门名是否包含模糊模式
f_all_enrich_count = f_all_enrich_count[~f_all_enrich_count['Phylum'].apply(lambda x: bool(ambiguous_patterns.search(str(x))))]
#  绘制气泡图 
# 定义分组与颜色之间的映射关系
f_group_colors = {
    'Cores': '#2471A3', # 蓝色
    'Uniques': '#D4AC0D',      # 金色
    'Generalists': '#EC7063',    # 红色
    'Specialists': '#82E0AA'     # 浅绿色
}
# 设置 Matplotlib 图的尺寸（宽度，高度）
plt.rcParams["figure.figsize"] = (5, 5)
# 使用 seaborn 绘制散点图（作为气泡图）
bubble_plot = sns.scatterplot(
    data= f_all_enrich_count,         # 输入数据 DataFrame
    x='Enrichment index',          # X轴：富集指数
    y='Phylum',                    # Y轴：门名称
    size='Count',                  # 气泡大小由 'Count'（OTU数量）决定
    hue='Ecotype',                 # 气泡颜色由 'Ecotype'（分组）决定
    palette=f_group_colors,          # 使用预定义的颜色映射
    sizes=(20, 250),               # 气泡大小的范围 (最小值, 最大值)
    alpha=1,                       # 气泡透明度
    edgecolor='black',             # 气泡边缘颜色
    linewidth=0.5                  # 气泡边缘线宽
)
# 添加垂直参考线，表示富集指数的阈值
# 通常，富集指数大于 2 或小于 -2 被认为是显著的
plt.axvline(x=-2, color='grey', linestyle='--') # 负富集（稀释）阈值
plt.axvline(x=2, color='grey', linestyle='--')  # 正富集阈值
# 设置图表标题、X轴和Y轴标签
plt.title('Fungal phyla representation within ecotype', size = 13)
plt.xlabel('Enrichment index', size = 13)
plt.ylabel('Phylum', size = 13)
# 调整图例位置，使其不与图表内容重叠
# bbox_to_anchor=(1.45, 0.63): 将图例锚点移动到轴外部
plt.legend(bbox_to_anchor=(1.45, 0.63))
# 设置X轴的显示范围
plt.xlim(-8.2,8.2)
# 定义保存图表的输出目录
output_dir = '/mnt/d/study//'
# 如果目录不存在，则创建它
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建输出目录: {output_dir}")
# 保存图表
# bbox_inches='tight' 确保所有图表元素都包含在保存的图像中
# dpi=600 设置图像的分辨率
plt.savefig(os.path.join(output_dir, 'ecotype_phyla_enrich_fungi.png'), bbox_inches='tight', dpi=600)
# 显示图表
plt.show()

